# Silver Layer — Clean & Conform Retail Data
Deduplicate transactions, conform dimensions, calculate line-level totals.

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, date_format,
    hour, dayofweek, row_number, current_timestamp,
    round as _round, trim, lower
)
from pyspark.sql.window import Window

In [ ]:
# Clean POS transactions
df_txn = spark.read.format('delta').table('bronze_pos_transactions')

# Deduplicate on transaction_id + product_id
w = Window.partitionBy('transaction_id', 'product_id').orderBy(col('ingestion_timestamp').desc())
df_txn = (
    df_txn
    .withColumn('_rn', row_number().over(w))
    .filter(col('_rn') == 1)
    .drop('_rn')
)

# Cast and derive
df_txn = (
    df_txn
    .withColumn('transaction_timestamp', to_timestamp('transaction_timestamp'))
    .withColumn('quantity', col('quantity').cast('int'))
    .withColumn('unit_price', col('unit_price').cast('double'))
    .withColumn('discount_pct', col('discount_pct').cast('double'))
    .filter(col('quantity') > 0)
    .filter(col('unit_price') > 0)
    # Calculate line total
    .withColumn('line_total',
                _round(col('quantity') * col('unit_price') * (1 - col('discount_pct') / 100), 2))
    .withColumn('transaction_date', date_format('transaction_timestamp', 'yyyy-MM-dd'))
    .withColumn('transaction_hour', hour('transaction_timestamp'))
    .withColumn('day_of_week', dayofweek('transaction_timestamp'))
    .withColumn('time_of_day',
                when(hour('transaction_timestamp') < 12, 'Morning')
                .when(hour('transaction_timestamp') < 17, 'Afternoon')
                .otherwise('Evening'))
    .withColumn('silver_timestamp', current_timestamp())
)

df_txn.write.mode('overwrite').format('delta').saveAsTable('silver_pos_transactions')
print(f'Silver POS transactions: {df_txn.count()} rows')

In [ ]:
# Clean product catalog
df_products = spark.read.format('delta').table('bronze_products')
df_products = (
    df_products
    .withColumn('product_name', trim(col('product_name')))
    .withColumn('category', trim(col('category')))
    .withColumn('subcategory', trim(col('subcategory')))
    .withColumn('brand', trim(col('brand')))
    .withColumn('unit_cost', col('unit_cost').cast('double'))
    .filter(col('sku').isNotNull())
    .withColumn('silver_timestamp', current_timestamp())
)

df_products.write.mode('overwrite').format('delta').saveAsTable('silver_products')
print(f'Silver products: {df_products.count()} rows')

In [ ]:
# Clean store locations
df_stores = spark.read.format('delta').table('bronze_stores')
df_stores = (
    df_stores
    .withColumn('store_name', trim(col('store_name')))
    .withColumn('city', trim(col('city')))
    .withColumn('state', trim(col('state')))
    .withColumn('region', trim(col('region')))
    .withColumn('store_format', trim(col('store_format')))
    .filter(col('store_id').isNotNull())
    .withColumn('silver_timestamp', current_timestamp())
)

df_stores.write.mode('overwrite').format('delta').saveAsTable('silver_stores')
print(f'Silver stores: {df_stores.count()} rows')

In [ ]:
# Clean inventory snapshots
df_inv = spark.read.format('delta').table('bronze_inventory_snapshots')
df_inv = (
    df_inv
    .withColumn('snapshot_date', to_timestamp('snapshot_date').cast('date'))
    .withColumn('quantity_on_hand', col('quantity_on_hand').cast('int'))
    .withColumn('quantity_on_order', col('quantity_on_order').cast('int'))
    .withColumn('reorder_point', col('reorder_point').cast('int'))
    .filter(col('quantity_on_hand') >= 0)
    .withColumn('below_reorder', col('quantity_on_hand') < col('reorder_point'))
    .withColumn('silver_timestamp', current_timestamp())
)

df_inv.write.mode('overwrite').format('delta').saveAsTable('silver_inventory_snapshots')
print(f'Silver inventory snapshots: {df_inv.count()} rows')